# Practica Spark DataFrame: Walmart Stock

Dataset utilizado:

https://raw.githubusercontent.com/pratikbarjatya/spark-walmart-data-analysis-exercise/master/walmart_stock.csv

Objetivo: practicar operaciones basicas y agregaciones con Spark DataFrames.

## Celda opcional para Google Colab

Ejecutar solo si el entorno no tiene Spark instalado.

In [ ]:
# Descomentar en Google Colab si hace falta instalar Spark
# !apt-get update -qq
# !apt-get install -y openjdk-17-jdk-headless -qq
# !pip install -q --force-reinstall pyspark==4.0.1
# import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# os.environ["PATH"] += ":/usr/lib/jvm/java-17-openjdk-amd64/bin"

## 1. Crear la SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, corr, format_number, max, mean, min, month, year

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Practica Spark DataFrame - Walmart Stock")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark.range(10).show()

## 2. Leer el CSV

In [ ]:
DATASET_PATH = "walmart_stock.csv"

df = spark.read.csv(DATASET_PATH, header=True, inferSchema=True)
df.show(5)

## 3. Inspeccion inicial

In [ ]:
df.columns

In [ ]:
df.printSchema()

In [ ]:
df.describe().show()

## 4. Resumen estadistico formateado

In [ ]:
result = df.describe()

result.select(
    result["summary"],
    format_number(result["Open"].cast("float"), 2).alias("Open"),
    format_number(result["High"].cast("float"), 2).alias("High"),
    format_number(result["Low"].cast("float"), 2).alias("Low"),
    format_number(result["Close"].cast("float"), 2).alias("Close"),
    result["Volume"].cast("int").alias("Volume"),
).show()

## 5. Crear la columna HV Ratio

`HV Ratio` representa la relacion entre el precio maximo del dia (`High`) y el volumen negociado (`Volume`).

In [ ]:
df_hv = df.withColumn("HV Ratio", df["High"] / df["Volume"])
df_hv.select("Date", "High", "Volume", "HV Ratio").show(10)

## 6. Dia con el precio High mas alto

In [ ]:
df.orderBy(df["High"].desc()).select("Date", "High").show(1)

## 7. Media de Close

In [ ]:
df.select(mean("Close").alias("Mean Close")).show()

## 8. Maximo y minimo de Volume

In [ ]:
df.select(
    max("Volume").alias("Max Volume"),
    min("Volume").alias("Min Volume"),
).show()

## 9. Dias con Close menor que 60

In [ ]:
df.filter(df["Close"] < 60).count()

## 10. Porcentaje de dias con High mayor que 80

In [ ]:
total_days = df.count()
high_days = df.filter(df["High"] > 80).count()

percentage = (high_days / total_days) * 100
percentage

## 11. Correlacion entre High y Volume

In [ ]:
df.select(corr("High", "Volume").alias("Corr High Volume")).show()

## 12. Maximo High por ano

In [ ]:
df.withColumn("Year", year("Date")) \
  .groupBy("Year") \
  .max("High") \
  .orderBy("Year") \
  .show()

## 13. Media de Close por mes

In [ ]:
df.withColumn("Month", month("Date")) \
  .groupBy("Month") \
  .avg("Close") \
  .orderBy("Month") \
  .show()

## 14. Media de Close por mes formateada

In [ ]:
df.withColumn("Month", month("Date")) \
  .groupBy("Month") \
  .agg(avg("Close").alias("Avg Close")) \
  .orderBy("Month") \
  .select("Month", format_number("Avg Close", 2).alias("Avg Close")) \
  .show()

## Conclusion

Se han practicado las operaciones principales de Spark DataFrames sobre un dataset bursatil: lectura de datos, inspeccion de esquema, seleccion de columnas, creacion de columnas derivadas, filtros, agregaciones, funciones de fecha y correlacion.